# 작물 추천 — EDA와 모델 학습 (Crop Recommendation)

- 데이터: 에티오피아 농지의 토양·기후 (3,867행 · 29열)
- 목표: 적합 작물 추천 (12종) — 다중분류
- 흐름: 불러오기 → 학습 전 확인 → EDA → 학습 → 해석
- 참고: 데이터 소개 data_CropRecommendation.txt

- 이 데이터는 **함정이 가장 많다**
  (정리 안 된 범주 · 지역 중복 · 불균형 · 순환변수)
  → 학습 전 확인이 특히 중요

## 1. 불러오기

- 일반 CSV(쉼표) · 결측 없음
- 컬럼명에 하이픈(QV2M-W 등) → 대괄호로 참조

In [ ]:
import pandas as pd
import numpy as np
from autogluon.common.utils.resource_utils import ResourceManager
ResourceManager.get_cpu_count = staticmethod(lambda **kwargs: 16)

df = pd.read_csv("Crop Recommendation using Soil Properties and Weather Prediction.csv")
print(df.shape)          # (3867, 29)
df.head()

## 2. 학습 전 확인 — 함정 네 가지

- 결측·인코딩은 AutoGluon이 자동
- 이 데이터는 사람이 볼 것이 유독 많다

### 2-1. Soilcolor 표기가 정리 안 됨 (가장 중요)

- 고유값 45개지만 실제 색은 훨씬 적음
- 대소문자("brown"/"Brown"), 오타, 세미콜론 부가정보가 섞임
- 그냥 두면 같은 색이 여러 범주로 쪼개져 학습됨

In [ ]:
print("정리 전 고유값:", df["Soilcolor"].nunique())   # 45

# 소문자·공백정리·세미콜론 앞부분만
df["Soilcolor"] = (df["Soilcolor"].str.lower().str.strip()
                   .str.split(";").str[0]
                   .str.replace(r"\s+", " ", regex=True))

print("정리 후 고유값:", df["Soilcolor"].nunique())   # 30
# 이후 오타·유사색 추가 통합은 판단해서

### 2-2. 기후 21열이 실제로는 16개 지역뿐

- 같은 지역 농지는 기후값이 완전히 동일
- 무작위 분할 시 같은 지역이 train·test 양쪽에 → 성능 과대평가
- **지역 단위로 분할**해야 정직한 성능

In [ ]:
# 기후 컬럼으로 지역 식별
w = [c for c in df.columns if any(k in c for k in
     ["QV2M","T2M","PRECTOT","WD10M","GWETTOP","CLOUD","WS2M","PS"])]
df["region"] = df.groupby(w, dropna=False).ngroup()
print("지역 수:", df["region"].nunique())   # 16

# 지역 단위 분할
ids = df["region"].unique()
rng = np.random.RandomState(42)
train_ids = rng.choice(ids, size=int(len(ids)*0.75), replace=False)
train_data = df[df["region"].isin(train_ids)].drop(columns=["region"])
test_data  = df[~df["region"].isin(train_ids)].drop(columns=["region"])
print("train:", train_data.shape, "test:", test_data.shape)

### 2-3. 풍향(WD10M)은 순환 변수

- 0도와 354도는 실제로 인접인데 숫자로는 가장 멀다
- 삼각함수로 변환해 순환성을 살림

In [ ]:
for d in [train_data, test_data]:
    rad = np.deg2rad(d["WD10M"])
    d["WD_sin"] = np.sin(rad)
    d["WD_cos"] = np.cos(rad)
    d.drop(columns=["WD10M"], inplace=True)
print("풍향 → WD_sin, WD_cos 로 변환")

### 2-4. 작물 불균형 (Teff 1260 vs Fallow 26, 48배)

- 정확도만 보면 소수 작물을 다 틀려도 점수가 높음
- balanced_accuracy로 평가해야 소수 작물도 반영

In [ ]:
print(df["label"].value_counts())
# → 학습 시 eval_metric="balanced_accuracy" 사용 (아래 4절)

## 3. EDA

In [ ]:
from data_profiling import ProfileReport

profile = ProfileReport(df, progress_bar=False)
profile.to_file("crop_eda.html")   # 브라우저에서 열기

### 3-1. 작물 분포 — 불균형 확인

In [ ]:
import matplotlib.pyplot as plt
df["label"].value_counts().plot(
    kind="bar", figsize=(10,3), title="crop counts (imbalanced)")
plt.show()

### 3-2. 토양 성분과 작물

- 작물별 토양 산도(Ph)·질소(N) 차이가 있는가

In [ ]:
df.groupby("label")[["Ph","N","K"]].mean().round(2)

## 4. 학습

- 불균형 대응 → eval_metric="balanced_accuracy"
- 결측·인코딩·범주 처리는 내부 자동

In [ ]:
from autogluon.tabular import TabularPredictor

predictor = TabularPredictor(
    label="label",
    eval_metric="balanced_accuracy",
).fit(
    train_data,
    presets="medium_quality",
    time_limit=300,
)

In [ ]:
predictor.leaderboard(test_data)

## 5. 해석

In [ ]:
perf = predictor.evaluate(test_data)
print(perf)

### 5-1. 변수 중요도

- 토양 성분과 기후 중 무엇이 작물을 가르는가

In [ ]:
predictor.feature_importance(test_data)

### 5-2. 지역 단위 분할의 효과 (선택 실험)

- 무작위 분할로 학습하면 성능이 더 높게 나올 것
- 하지만 그것은 같은 지역이 양쪽에 든 '과대평가'
- 지역 분할이 진짜 성능 → 두 방식을 비교해 볼 것

In [ ]:
# (참고) 무작위 분할 버전과 성능을 비교하면
# 무작위 쪽이 비현실적으로 높게 나오는 것을 확인할 수 있다
# → 데이터 구조를 모르면 성능을 착각하게 된다

## 정리

- Soilcolor 정리 → 같은 색이 쪼개지지 않게 (사람의 판단)
- 기후 16지역 → 지역 단위 분할 (누수 방지)
- 풍향 → 삼각함수 변환 (순환변수)
- 불균형 → balanced_accuracy
- 학습·앙상블은 TabularPredictor가 자동

- 결측 0건이지만 '깨끗한' 데이터가 아니다
  → 함정은 EDA와 도메인 이해로 직접 찾아야 한다